In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts" / "edonna").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts" / "edonna"))
load_dotenv(REPO_ROOT / "platform_edu" / ".env")

if not os.environ.get("EDONNA_API_KEY"):
    raise RuntimeError("Set EDONNA_API_KEY in platform_edu/.env")

In [ ]:
import pandas as pd

from customer_data import build_customer_summary
from daily_lists import generate_daily_lists

customer_summary = build_customer_summary()
daily_lists = generate_daily_lists(customer_summary)

print(customer_summary.head(20))
print("\nTotal users:", len(customer_summary))
print("Never ordered:", customer_summary["never_ordered"].sum())
print("Inactive 30d:", customer_summary["inactive_30d"].sum())
print("New users:", customer_summary["is_new_user"].sum())

for name, frame in daily_lists.items():
    print(f"\n{name}: {len(frame)}")
    display(frame.head(10))

In [ ]:
import requests
import pandas as pd

BREVO_API_KEY = "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"
LIST_ID = 11

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "api-key": BREVO_API_KEY,
}

inactive_df = customer_summary[
    customer_summary["inactive_30d"] == True
]

uploaded = 0

for _, row in inactive_df.iterrows():

    if pd.isna(row["email"]):
        continue

    payload = {
        "email": row["email"],
        "attributes": {
            "FIRSTNAME": row["user_name"],
            "TOTAL_ORDERED": float(row["total_ordered"]),
            "ORDER_COUNT": int(row["order_count"]),
            "DAYS_SINCE_LAST_ORDER": int(row["days_since_last_order"]),
        },
        "updateEnabled": True,
        "listIds": [LIST_ID],
    }

    response = requests.post(
        "https://api.brevo.com/v3/contacts",
        json=payload,
        headers=headers,
        timeout=30,
    )

    if response.status_code in (200, 201, 204):
        uploaded += 1
    else:
        print(
            f"Failed: {row['email']} | "
            f"{response.status_code} | {response.text}"
        )

print(f"Uploaded {uploaded} contacts to list {LIST_ID}")